# Demostración de Reutilización del Modelo
## XGBoost — Clasificación de Severidad de Accidentes de Tránsito
### ATUS 2024 — Proyecto Final — Almacenes y Minería de Datos

Este notebook demuestra que el modelo entrenado puede cargarse y utilizarse para
realizar predicciones nuevas de forma completamente independiente del notebook de
entrenamiento, cumpliendo con el requisito de persistencia del proyecto.

El flujo completo incluye: carga del dataset limpio, preprocesamiento mediante las
clases desarrolladas en `src/`, carga del modelo serializado y predicción con el
umbral ajustado óptimo (t=0.15).

In [1]:
import sys
sys.path.append('../src/supervised/')

from data_repository import DataRepository
from preprocessor import Preprocessor
from trainer import Trainer
from evaluator import Evaluator

#### Configuración Global

In [2]:
RANDOM_STATE = 42 # Misma que el notebook de modelado
THRESHOLD    = 0.15 # Umbral de decisión óptimo para el modelo

### **1. Carga del Dataset**

In [3]:
repo = DataRepository("../data/atus_anual_2024_limpio.csv")
df   = repo.load()
print(repo.summary())

           RESUMEN DEL DATASET CARGADO
  Ruta:          ..\data\atus_anual_2024_limpio.csv
  Configuración: C:\Users\grame\Desktop\Tareas_Almacenes-y-Mineria-de-Datos\ProyectoFinal\notebooks\..\src\supervised\config\repository_config.yaml
  Filas:         374,078
  Columnas:      42

  Distribución de CLASACC (variable objetivo):
    Sólo daños   306,088  (81.82%)
    No fatal      63,809  (17.06%)
    Fatal          4,181  (1.12%)

  Valores nulos por columna:
    ID_EDAD                    96,222


### **2. Preprocesamiento**

In [4]:
y = df[repo.target_column]
X = df.drop(columns=[repo.target_column])

prep          = Preprocessor()
X_transformed = prep.fit_transform(X)
y_enc         = prep.fit_transform_target(y)

print(prep.summary())

Clases codificadas:
  0 → Fatal
  1 → No fatal
  2 → Sólo daños
        CONFIGURACIÓN DEL PREPROCESADOR
  Configuración: C:\Users\grame\Desktop\Tareas_Almacenes-y-Mineria-de-Datos\ProyectoFinal\notebooks\..\src\supervised\config\preprocessor_config.yaml

  Columnas excluidas      (17):
    - ID_MINUTO
    - ID_DIA
    - NOM_MUN
    - TRANVIA
    - FERROCARRI
    - OTROVEHIC
    - TOTAL_MUERTOS
    - CONDMUERTO
    - CONDHERIDO
    - PASAMUERTO
    - PASAHERIDO
    - PEATMUERTO
    - PEATHERIDO
    - CICLMUERTO
    - CICLHERIDO
    - OTROMUERTO
    - OTROHERIDO

  Numéricas               (11): Imputer(mediana) + StandardScaler
    - ID_EDAD
    - AUTOMOVIL
    - CAMPASAJ
    - MICROBUS
    - PASCAMION
    - OMNIBUS
    - CAMIONETA
    - CAMION
    - TRACTOR
    - MOTOCICLET
    - BICICLETA

  Binarias                (2): passthrough
    - CONDUCTOR_FUGADO
    - EDAD_DESCONOCIDA

  Categóricas             (10): OneHotEncoder(drop='first')
    - DIASEMANA
    - URBANA
    - SUBURBANA
    

### **3. Carga del Modelo**

In [5]:
THRESHOLD = 0.15

model = Trainer.load_model('xgboost', output_dir='../models')
print(f"\nUmbral de clasificación para Fatal: {THRESHOLD}")

[xgboost] Modelo cargado desde: ..\models\xgboost.joblib

Umbral de clasificación para Fatal: 0.15


### **4. Predicción sobre Registros**

Se seleccionan 25 registros del dataset como ejemplo de predicción en producción.
Para cada registro se muestran las probabilidades estimadas por clase y la predicción
final aplicando el umbral ajustado.

In [6]:
sample_df          = df.drop(columns=[repo.target_column]).sample(25, random_state=RANDOM_STATE)
sample_transformed = prep.transform(sample_df)
sample_probs       = model.predict_proba(sample_transformed)

evaluator      = Evaluator(prep.target_classes)
sample_probs   = model.predict_proba(sample_transformed)
sample_pred    = evaluator.predict_with_threshold(sample_probs, THRESHOLD)
sample_labels  = prep.inverse_transform_target(sample_pred)

print(f"{'#':<4} {'Fatal':>8} {'No fatal':>10} {'Sólo daños':>12} {'Predicción':<15} {'Real'}")
print("-" * 65)
for i, (probs, label) in enumerate(zip(sample_probs, sample_labels)):
    real = df[repo.target_column].iloc[sample_df.index[i]]
    print(
        f"{i+1:<4} {probs[0]:>8.3f} {probs[1]:>10.3f} {probs[2]:>12.3f} "
        f"{label:<15} {real}"
    )

#       Fatal   No fatal   Sólo daños Predicción      Real
-----------------------------------------------------------------
1       0.000      0.016        0.984 Sólo daños      Sólo daños
2       0.001      0.030        0.969 Sólo daños      Sólo daños
3       0.000      0.011        0.988 Sólo daños      Sólo daños
4       0.000      0.015        0.985 Sólo daños      Sólo daños
5       0.012      0.143        0.845 Sólo daños      Sólo daños
6       0.019      0.705        0.276 No fatal        Sólo daños
7       0.042      0.230        0.728 Sólo daños      Sólo daños
8       0.002      0.025        0.973 Sólo daños      Sólo daños
9       0.012      0.402        0.586 Sólo daños      Sólo daños
10      0.001      0.212        0.787 Sólo daños      Sólo daños
11      0.012      0.515        0.473 No fatal        No fatal
12      0.002      0.091        0.907 Sólo daños      Sólo daños
13      0.040      0.421        0.539 Sólo daños      Sólo daños
14      0.003      0.017        

## **5. Verificación del Pipeline Completo**

La demostración confirma que el pipeline completo funciona de forma independiente:
las clases `DataRepository`, `Preprocessor` y `Trainer` reproducen exactamente el
mismo flujo de preprocesamiento y predicción del notebook de entrenamiento, sin
necesidad de acceder a ninguna variable de sesión previa.

El umbral ajustado (t=0.15) se aplica manualmente sobre las probabilidades de la
clase `Fatal`, replicando el comportamiento optimizado identificado durante la
evaluación del modelo.